# Global spin-up on a GPU

Produce a **spun-up global atmospheric state** for JAX-GCM, which the repository
does not currently contain, and which blocks every free-running global
experiment — including convective precipitation maps.

### Why this is needed

`jcm/data/test/echam_t63l47/spinup_state.nc` is not a spun-up state despite its
name: it is the dycore's `isothermal_rest_atmosphere` default saved to disk —
288 K at every level, zero wind, zero humidity. An isothermal atmosphere is
*absolutely stable*, so no convection scheme can fire in it, and with no wind
the surface barely evaporates. Measured on CPU, a run from it moistens at
0.002 kg/m²/day against ~25 kg/m² for a real atmosphere, and the increments
*decelerate* — it is settling into a dry equilibrium, not spinning up.

The physical case for breaking out of that is: prescribed SSTs impose a
meridional temperature gradient, the orography breaks zonal symmetry so
stationary waves can grow, and grey radiation drives toward a state that does
have a lapse rate. The expected timescale is tens of simulated days, which is
unaffordable on CPU (~11 min per simulated day) and cheap on a GPU.

**This is a long shot made cheap, not a certainty.** The 10-day go/no-go in the
diagnostics cell is there so you spend one session finding out, not ten.

### What it does

* spins up with **Tiedtke**, not the GISS port — what needs spinning up is the
  atmosphere, so use the fast, well-tested scheme; the resulting state is then
  reusable for any scheme, permanently
* runs in **segments** with a checkpoint to Google Drive after each, so a Colab
  session timeout costs one segment rather than the whole run
* appends one row per segment to a CSV on Drive, so the go/no-go record survives
  a lost session

### Expected cost

The repository's own harness notes ~30 min for 10 simulated days on a GPU. A
free T4 may be 3–5× slower than whatever that was measured on, so budget
several sessions for 90 days. Pro (L4/A100) should manage it in one or two.

## 1. Confirm you actually have a GPU

**Runtime → Change runtime type → T4 GPU** (or L4/A100 on Pro) before running
this. If this cell prints no GPU table, stop and fix the runtime — everything
below will silently fall back to CPU and be unusably slow.

In [ ]:
!nvidia-smi || echo "NO GPU -- Runtime > Change runtime type > GPU, then rerun"


## 2. Clone the repository

The data the run needs (terrain, forcing, the stub state — 18 MB) is committed
in the repo, so there is nothing else to upload.

**Push your local commits first.** This clones from GitHub, so anything
uncommitted or unpushed locally will not be here.

If your fork is **private**, create a fine-grained personal access token with
read access to it (github.com → Settings → Developer settings → Personal access
tokens) and paste it when prompted. If it is public, just press Enter.

In [ ]:
import getpass, os, subprocess

REPO_OWNER = "abaghyangor"
REPO_NAME  = "jax-gcm"
BRANCH     = "modele-convection-port"   # needs the mixed_layer_top + precip commits

token = getpass.getpass("GitHub token (press Enter if the repo is public): ").strip()
auth = f"{token}@" if token else ""
url = f"https://{auth}github.com/{REPO_OWNER}/{REPO_NAME}.git"

if not os.path.isdir("/content/jax-gcm"):
    # Shallow, single-branch: the full history is ~227 MB and none of it is needed.
    r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        url, "/content/jax-gcm"],
                       capture_output=True, text=True)
    print(r.stdout[-2000:]); print(r.stderr[-2000:])
    if r.returncode:
        raise SystemExit("clone failed -- check the branch name, and the token if private")
%cd /content/jax-gcm
!git log --oneline -3
!ls -la jcm/data/bc/t63/ jcm/data/test/echam_t63l47/


## 3. Install

Order matters. `pip install -e .` pulls the package's own requirements with an
unpinned `jax`, which on Colab can replace the CUDA build with a CPU-only one —
so the CUDA wheel is installed **last** and wins.

Expect pip to complain about dependency conflicts with Colab's preinstalled
packages. Those are normally harmless here; the verification cell is what
decides.

In [ ]:
%cd /content/jax-gcm
!pip install -q -e . 2>&1 | tail -5
!pip install -q -U "jax[cuda12]" 2>&1 | tail -5
print("\nIf jax was reinstalled, Colab may need a restart -- see the next cell.")


## 4. Verify the GPU is visible to JAX

**If this prints `CpuDevice`**, do: *Runtime → Restart session*, then re-run
**only** this cell and everything after it — skip the install cell, the packages
are already there. This is the single most common way this notebook goes wrong:
a CPU fallback that runs ~30× slower and looks like it is working.

In [ ]:
import os
# Must be set before JAX initialises. Colab shares the GPU with the notebook
# process; preallocating the whole card tends to end in an OOM on a T4.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.85"

import jax
print("jax", jax.__version__)
print("devices:", jax.devices())
print("backend:", jax.default_backend())
assert jax.default_backend() == "gpu", (
    "JAX is not on the GPU. Restart the session (Runtime > Restart session), "
    "then re-run from this cell. Do not re-run the install cell.")


## 5. Mount Drive for checkpoints

Checkpoints and the diagnostics log go to Drive so they outlive the session.
Without this a timeout loses the entire run.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
CKPT_DIR = "/content/drive/MyDrive/jcm_spinup"
os.makedirs(CKPT_DIR, exist_ok=True)
print("checkpoints ->", CKPT_DIR)
print(sorted(os.listdir(CKPT_DIR)) or "(empty -- this is a fresh start)")


## 6. Configuration

`SEGMENT_DAYS = 10` is deliberately the go/no-go unit: run one segment, read the
diagnostics, and only continue if column water vapour is *climbing* rather than
flattening.

Leave `SCHEME = "tiedtke"` for the spin-up. Switch to `"giss"` only once you
have a spun-up state and want the port's precipitation.

In [ ]:
import logging

TOTAL_DAYS   = 90        # simulated days to aim for
SEGMENT_DAYS = 10        # simulated days per checkpoint
DT_MINUTES   = 12.0      # the production T63L47 timestep (minutes, not seconds)
TRUNC, LEVELS = 63, 47
SCHEME       = "tiedtke" # "tiedtke" to spin up; "giss" for the ported scheme
BC           = "/content/jax-gcm/jcm/data/bc/t63"
STUB         = "/content/jax-gcm/jcm/data/test/echam_t63l47/spinup_state.nc"
LOG_CSV      = os.path.join(CKPT_DIR, "spinup_log.csv")


## 7. Model, checkpoint and diagnostic helpers

Checkpoints are written in the same layout as `spinup_state.nc`, so anything
that reads that file reads these too.

Two details worth knowing rather than discovering:

* `PhysicsState` carries specific humidity in **g/kg**, and its arrays are
  ordered **top-first** (index 0 is the model top). These files are written that
  way. The committed stub stores its `level` coordinate reversed, which is
  invisible there only because every field in it is uniform.
* A checkpoint saves the prognostic state but **not** the physics carry (TKE,
  sub-cycled radiation). Crossing a segment boundary re-seeds it. That is
  harmless over 10-day segments, but it is why the loop below uses `resume`
  within a session and only reloads from disk when a session has been lost.

In [ ]:
import time
import numpy as np
import xarray as xr

from jcm.model import Model
from jcm.terrain import TerrainData
from jcm.forcing import ForcingData
from jcm.utils import get_coords, load_states_from_xarray
from jcm.physics.echam.echam_levels import get_echam_levels
from jcm.physics.echam.echam_terms import echam_physics
from jcm.physics.dissipation import UpperSponge
from jcm.constants import p0 as P0_PA

GRAV = 9.80665


def build_model(scheme=SCHEME):
    coords = get_coords(get_echam_levels(LEVELS), spectral_truncation=TRUNC)
    terrain = TerrainData.from_file(f"{BC}/terrain.nc", coords=coords)
    forcing = ForcingData.from_file(f"{BC}/forcing.nc", coords=coords)
    physics = echam_physics(radiation_scheme="grey") + UpperSponge(
        n_sponge_levels=5, sponge_timescale_s=3 * 3600.0, enspodi=2.0)
    if scheme == "giss":
        from jcm.physics.convection.giss_mstcnv import GissConvection
        physics = physics.replace("convection",
                                  GissConvection(allow_mc=True, bsort=True))
    model = Model(coords=coords, terrain=terrain, physics=physics,
                  time_step=DT_MINUTES, log_level=logging.CRITICAL)
    return model, coords, forcing


def read_state(path):
    ds = xr.open_dataset(path)
    tracers = {t: t for t in ("qc", "qi") if t in ds.data_vars}
    return load_states_from_xarray(ds, tracer_vars=tracers or None)


def save_checkpoint(model, coords, day):
    """Write the current prognostic state in `spinup_state.nc`'s layout."""
    # `_final_dycore_state` is how Model itself carries the state between
    # run/resume calls; `to_physics_state` is the public conversion.
    ps = model.dycore.to_physics_state(model._final_dycore_state)
    g = lambda a: np.asarray(a)
    lat = np.degrees(np.arcsin(np.clip(
        np.asarray(coords.horizontal.nodal_axes[1]), -1, 1)))
    lon = np.degrees(np.asarray(coords.horizontal.nodal_axes[0]))
    level = np.asarray(coords.vertical.get_sigma_centers(P0_PA))
    d3 = ("level", "lon", "lat")
    tr = ps.tracers or {}
    zero = np.zeros_like(g(ps.temperature))
    out = xr.Dataset(
        {"u_wind": (d3, g(ps.u_wind).astype("float32")),
         "v_wind": (d3, g(ps.v_wind).astype("float32")),
         "temperature": (d3, g(ps.temperature).astype("float32")),
         "specific_humidity": (d3, g(ps.specific_humidity).astype("float32")),
         "geopotential": (d3, g(ps.geopotential).astype("float32")),
         "normalized_surface_pressure":
             (("lon", "lat"), g(ps.normalized_surface_pressure).astype("float32")),
         "qc": (d3, g(tr.get("qc", zero)).astype("float32")),
         "qi": (d3, g(tr.get("qi", zero)).astype("float32"))},
        coords={"level": level, "lon": lon, "lat": lat})
    out["specific_humidity"].attrs["units"] = "g/kg"
    out.attrs["description"] = (
        f"JAX-GCM T{TRUNC}L{LEVELS} spin-up, {SCHEME} convection, "
        f"{day} simulated days from the isothermal stub. Arrays top-first; "
        "specific humidity in g/kg.")
    path = os.path.join(CKPT_DIR, f"spinup_day{day:04d}.nc")
    out.to_netcdf(path)
    return path


def latest_checkpoint():
    """Highest-numbered checkpoint on Drive, or (0, None) for a fresh start."""
    best = (0, None)
    for f in os.listdir(CKPT_DIR):
        if f.startswith("spinup_day") and f.endswith(".nc"):
            try:
                d = int(f[len("spinup_day"):-3])
            except ValueError:
                continue
            if d >= best[0]:
                best = (d, os.path.join(CKPT_DIR, f))
    return best


def segment_diagnostics(ds, coords):
    """Column water vapour and precipitation: the numbers the go/no-go rests on."""
    pick = lambda n: (np.asarray(ds[n].isel(time=-1).values)
                      if n in ds.data_vars else None)
    q = pick("specific_humidity")            # g/kg
    rho, dz = pick("air_density"), pick("layer_thickness")
    q_col = np.sum(q / 1000.0 * rho * dz, axis=0)       # kg/m^2
    lat = np.degrees(np.arcsin(np.clip(
        np.asarray(coords.horizontal.nodal_axes[1]), -1, 1)))
    w = np.broadcast_to(np.cos(np.radians(lat))[None, :], q_col.shape)
    area_mean = lambda f: float(np.sum(f * w) / np.sum(w))
    t = pick("temperature")
    rain = pick("clouds.precip_rain")
    snow = pick("clouds.precip_snow")
    conv = pick("convection.mcp")
    if conv is None:
        pc = pick("convection.precip_conv")
        conv = None if pc is None else pc * 86400.0      # kg/m2/s -> mm/day
    tot = 0.0
    for f in (rain, snow):
        if f is not None:
            tot += area_mean(f) * 86400.0
    return dict(q_col=area_mean(q_col),
                tmin=float(np.nanmin(t)), tmax=float(np.nanmax(t)),
                large_scale_precip=tot,
                conv_precip=(None if conv is None else area_mean(conv)),
                nan=int(np.isnan(t).sum()))


def append_log(row):
    import csv
    new = not os.path.exists(LOG_CSV)
    with open(LOG_CSV, "a", newline="") as fh:
        wr = csv.DictWriter(fh, fieldnames=list(row))
        if new:
            wr.writeheader()
        wr.writerow(row)

print("helpers ready")


## 8. Run

Re-runnable and resumable: it picks up from the newest checkpoint on Drive, so
after a disconnect just re-run cells 4 onward.

It stops immediately on a NaN rather than burning GPU hours on a dead run.

Keep the browser tab alive — Colab disconnects idle sessions after about 90
minutes. The per-segment printing helps, but do not close the laptop.

In [ ]:
model, coords, forcing = build_model(SCHEME)

day, ckpt = latest_checkpoint()
if ckpt:
    print(f"resuming from {os.path.basename(ckpt)} at day {day}")
    state = read_state(ckpt)
else:
    print("fresh start from the isothermal stub")
    state = read_state(STUB)

first_in_session = True
while day < TOTAL_DAYS:
    t0 = time.time()
    if first_in_session:
        # `run` seeds from an explicit state; `resume` continues and keeps the
        # physics carry, so it is used for every later segment in this session.
        preds = model.run(initial_state=state, forcing=forcing,
                          save_interval=float(SEGMENT_DAYS),
                          total_time=float(SEGMENT_DAYS))
        first_in_session = False
    else:
        preds = model.resume(forcing=forcing,
                             save_interval=float(SEGMENT_DAYS),
                             total_time=float(SEGMENT_DAYS))
    ds = preds.to_xarray()
    day += SEGMENT_DAYS
    wall = time.time() - t0

    d = segment_diagnostics(ds, coords)
    conv_txt = ('    --' if d['conv_precip'] is None
                else format(d['conv_precip'], '6.3f'))
    d.update(day=day, wall_s=round(wall, 1), scheme=SCHEME)
    cp = "" if d["nan"] else save_checkpoint(model, coords, day)
    append_log(d)
    print(f"day {day:4d}  {wall/60:5.1f} min  "
          f"q_col {d['q_col']:7.3f} kg/m2  "
          f"T {d['tmin']:6.1f}-{d['tmax']:6.1f} K  "
          f"ls_precip {d['large_scale_precip']:6.3f}  "
          f"conv {('--' if d['conv_precip'] is None else f'{d["conv_precip"]:6.3f}')} mm/day"
          f"  nan {d['nan']}")
    del preds, ds

    if d["nan"]:
        print("\nNaN in the state -- stopping. The last good checkpoint is intact.")
        break
else:
    print(f"\nreached {TOTAL_DAYS} days; newest checkpoint in {CKPT_DIR}")


## 9. The go/no-go

The question is whether **column water vapour is climbing or flattening**.

* **Climbing, increments steady or growing** → spin-up is working. Keep going to
  `TOTAL_DAYS`. A realistic atmosphere holds ~25 kg/m² in the global mean.
* **Flattening below ~1 kg/m², increments shrinking** → it is settling into the
  same dry equilibrium seen on CPU, and more days will not help. Stop, and get a
  reanalysis initial state instead.

On CPU the first three days gave 0.002, 0.004, 0.005 kg/m² — increments of
0.002, 0.002, 0.001. That is the failure signature, at three days. Ten days
should be unambiguous.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv(LOG_CSV)
print(log.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(log.day, log.q_col, "o-")
ax[0].set_xlabel("simulated day"); ax[0].set_ylabel("column water vapour [kg m$^{-2}$]")
ax[0].axhline(25, ls="--", c="0.6")
ax[0].text(0.02, 0.92, "~25 = realistic", transform=ax[0].transAxes,
           fontsize=8, color="0.4")
ax[0].set_title("is this climbing or flattening?", fontsize=10)

ax[1].plot(log.day, log.large_scale_precip, "o-", label="large-scale")
if log.conv_precip.notna().any():
    ax[1].plot(log.day, log.conv_precip, "s-", label="convective")
ax[1].set_xlabel("simulated day"); ax[1].set_ylabel("precipitation [mm day$^{-1}$]")
ax[1].legend(fontsize=8); ax[1].set_title("precipitation", fontsize=10)
fig.tight_layout(); plt.show()

if len(log) >= 2:
    inc = np.diff(log.q_col.values)
    print(f"\nincrements per segment: {np.round(inc, 4)}")
    verdict = ("CLIMBING -- continue" if inc[-1] > 0.5 * max(inc[0], 1e-9)
               and log.q_col.values[-1] > 1.0
               else "FLATTENING -- likely the dry equilibrium; consider reanalysis")
    print("verdict:", verdict)


## 10. Once spun up: the GISS port and its maps

Only worth running after the diagnostics above say the spin-up worked.

Set `SCHEME = "giss"` in the configuration cell and re-run from cell 7 to carry
the ported scheme forward from the newest checkpoint.

Two caveats for anything plotted from this, both measured against ModelE run as
an oracle on single columns:

* the port's **column water budget** matches ModelE to 0.5%/0.4%/5% on
  BOMEX/RICO/TWP-ICE, which is the number to quote;
* its **surface precipitation** still has a heavy upper tail — typical periods
  agree within a few percent, but the mean runs 1.2–1.5× high. Label a
  precipitation map accordingly, and prefer the column moisture sink where a
  defensible magnitude matters.

In [ ]:
# Download the newest checkpoint to commit into the repository as a reusable
# initial state, once you are satisfied with it.
day, ckpt = latest_checkpoint()
print(f"newest checkpoint: {ckpt} (day {day})")
if ckpt:
    ds = xr.open_dataset(ckpt)
    print(ds)
    # from google.colab import files; files.download(ckpt)
